# TxSON MET Data Cleaning Pipeline — Part 2: Interactive Dashboard

**Part 2 of 2.** Run `txson_met_pipeline_1_processing.ipynb` first — this notebook
only *reads* the CSVs it wrote to `OUTPUT_DIR`; it does not recompute anything.
You can re-run this notebook as often as you like (e.g. after re-running Part 1
with new data) without touching Part 1's code.

Point `OUTPUT_DIR` below at the same folder Part 1 used.

**If the dashboard renders the dropdowns but the plot area stays completely
blank (no chart, no text, no error):** this is a known Colab issue where
`ipywidgets` needs Colab's custom widget manager explicitly enabled, which
the imports cell below now does automatically. If it's still blank after
re-running from the imports cell down, use *Runtime → Restart session* once
and run all cells again — a widget created before the manager was enabled
can stay blank even after the fix is in place.


In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
!pip install -q plotly ipywidgets

import pandas as pd
import numpy as np
import glob, os, calendar
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

# Colab-specific: newer ipywidgets can render a completely blank Output
# widget - no error, no content, just empty space below the dropdowns -
# unless Colab's custom widget manager is explicitly enabled. This is a
# one-line fix, safe to run even if it's not needed.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # not running in Colab

print('Libraries ready.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 69.7 MB/s eta 0:00:00
Libraries ready.


## Step 1 — Mount Drive & point to Part 1's output folder

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ▶ Must match OUTPUT_DIR from Part 1
OUTPUT_DIR = '/content/drive/MyDrive/DSResearch_TexasSoil2026/output'

COL_TS   = 'Date'
COL_RH   = 'RH'
MONTHS       = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

COLORS = {
    'valid': '#2ca02c', 'flagged': '#d62728', 'stuck': '#ff7f0e',
    'suspect': '#ffbb78', 'gap': '#7f7f7f', 'jump': '#9467bd',
}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Reload cleaned data written by Part 1

Loads `clean_{station}_met.csv` (Stage 4 output = `VALID_CLEAN`) and
`{station}_rh_monthly_clean.csv` (Stage 6 output = `MONTHLY_CLEAN`) for every
station found in `OUTPUT_DIR`. The audit CSVs (stuck periods, gaps, jumps) are
read directly by the plot functions below, so they don't need to be preloaded.


In [5]:
VALID_CLEAN   = {}
MONTHLY_CLEAN = {}

# Using os.listdir + manual filtering here instead of glob.glob - some
# network/FUSE-mounted filesystems (this can happen with Drive mounts) don't
# support glob's wildcard-aware directory scan and raise errors like
# "string pattern matching is not implemented", even though the files are
# genuinely present. os.listdir + a plain string check only needs a basic
# directory listing, which every backend supports.
all_files = sorted(os.listdir(OUTPUT_DIR))

clean_files = [os.path.join(OUTPUT_DIR, f) for f in all_files
              if f.startswith('clean_') and f.endswith('_met.csv')]
for fpath in clean_files:
    sid = os.path.basename(fpath).replace('clean_', '').replace('_met.csv', '')
    df = pd.read_csv(fpath, parse_dates=[COL_TS], index_col=COL_TS)
    VALID_CLEAN[sid] = df

monthly_files = [os.path.join(OUTPUT_DIR, f) for f in all_files
                if f.endswith('_rh_monthly_clean.csv')]
for fpath in monthly_files:
    sid = os.path.basename(fpath).replace('_rh_monthly_clean.csv', '')
    df = pd.read_csv(fpath, index_col='year')
    MONTHLY_CLEAN[sid] = df

ALL_STATIONS = sorted(set(VALID_CLEAN) | set(MONTHLY_CLEAN))

if not ALL_STATIONS:
    raise RuntimeError(
        f'No clean_*_met.csv or *_rh_monthly_clean.csv files found in {OUTPUT_DIR}.\n'
        f'Files actually present there: {all_files}\n'
        f'Run txson_met_pipeline_1_processing.ipynb first.'
    )

print(f'Loaded {len(ALL_STATIONS)} station(s): {ALL_STATIONS}')


Loaded 6 station(s): ['CB01', 'CB04', 'CB06', 'FD02', 'FD03', 'WC05']


## Stage 8 — Interactive dashboard

Dropdown-driven views: RH time series with stuck/flagged overlays, valid-vs-flagged
summary, gap timeline, seasonal heatmap, and jump/anomaly tables. All data comes
straight from disk — reflects exactly what Part 1 wrote.


In [6]:
# ── Plot functions ───────────────────────────────────────────────────────────

def plot_rh_timeseries(station):
    df = VALID_CLEAN.get(station)
    if df is None or COL_RH not in df.columns:
        print(f'No data for {station}'); return
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df.index, y=df[COL_RH], mode='lines',
                              name='RH (cleaned)', line=dict(color=COLORS['valid'], width=1)))

    audit_path = os.path.join(OUTPUT_DIR, f'audit_stuck_{station}.csv')
    if os.path.exists(audit_path):
        audit = pd.read_csv(audit_path, parse_dates=['start', 'end'])
        for _, r in audit.iterrows():
            fig.add_vrect(x0=r['start'], x1=r['end'],
                          fillcolor=COLORS['stuck'] if r['flag'] == 'STUCK' else COLORS['suspect'],
                          opacity=0.3, line_width=0)

    fig.update_layout(title=f'{station} — Cleaned RH time series (shaded = nulled stuck/suspect periods)',
                      xaxis_title='Date', yaxis_title='RH (%)', height=420)
    fig.show()


def plot_valid_vs_flagged(station):
    fpath = os.path.join(OUTPUT_DIR, f'flagged_{station}_met.csv')
    if not os.path.exists(fpath):
        print(f'No flagged-value log for {station}'); return
    flagged = pd.read_csv(fpath)
    if flagged.empty:
        print(f'{station}: no out-of-bounds values were flagged during prewash.'); return

    counts = flagged['variable'].value_counts()
    fig = go.Figure(go.Bar(x=counts.index.tolist(), y=counts.values.tolist(),
                           marker_color=COLORS['flagged']))
    fig.update_layout(title=f'{station} — Out-of-bounds values nulled during prewash, by variable',
                      xaxis_title='Variable', yaxis_title='Count', height=350)
    fig.show()
    print(f'{len(flagged)} total flagged values. Sample of original (now-nulled) values:')
    print(flagged.head(10).to_string(index=False))


def plot_gap_timeline(station):
    path = os.path.join(OUTPUT_DIR, f'{station}_gap_summary.csv')
    if not os.path.exists(path):
        print(f'No gap summary for {station}'); return
    gaps = pd.read_csv(path, parse_dates=['start', 'end'])
    fig = go.Figure()
    for cat, color in zip(['Short', 'Medium', 'Long', 'VeryLong'],
                          ['#c7e9c0', '#fdae6b', '#e6550d', '#a50f15']):
        sub = gaps[gaps['category'] == cat]
        if sub.empty: continue
        fig.add_trace(go.Bar(x=sub['duration_h'], y=sub['start'].astype(str),
                             orientation='h', name=cat, marker_color=color))
    fig.update_layout(title=f'{station} — Missing-data gaps by duration',
                      xaxis_title='Duration (hours)', height=max(350, 20*len(gaps)), barmode='overlay')
    fig.show()


def plot_seasonal_heatmap(station):
    df = MONTHLY_CLEAN.get(station)
    if df is None:
        print(f'No monthly data for {station}'); return
    fig = go.Figure(go.Heatmap(z=df[MONTHS].values, x=MONTH_LABELS, y=df.index.astype(str),
                               colorscale='YlGnBu', colorbar_title='RH %'))
    fig.update_layout(title=f'{station} — Monthly mean RH by year', height=420)
    fig.show()


def show_flag_summary(station):
    print(f'=== {station} — Flag summary ===\n')
    for fname, label in [
        (f'audit_stuck_{station}.csv', 'Stuck/suspect periods'),
        (f'{station}_gap_summary.csv', 'Missing-data gaps'),
    ]:
        path = os.path.join(OUTPUT_DIR, fname)
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f'{label}: {len(df)} rows')
        else:
            print(f'{label}: none')

    for fname, label, col in [
        ('audit_hourly_jumps.csv', 'Hourly jumps', 'station'),
        ('audit_monthly_within_year_jumps.csv', 'Monthly within-year jumps', 'station'),
        ('audit_monthly_cross_year_anomalies.csv', 'Cross-year anomalies', 'station'),
    ]:
        path = os.path.join(OUTPUT_DIR, fname)
        if os.path.exists(path):
            df = pd.read_csv(path)
            n = (df[col] == station).sum()
            print(f'{label}: {n} rows for {station}')

print('Plot functions defined.')


Plot functions defined.


In [9]:
# ── Dashboard wiring ─────────────────────────────────────────────────────────
station_dd = widgets.Dropdown(options=ALL_STATIONS, description='Station:')
view_dd = widgets.Dropdown(
    options=['RH time series', 'Valid vs flagged', 'Gap timeline',
             'Seasonal heatmap', 'Flag summary'],
    description='View:'
)
out = widgets.Output()

def render(*_):
    with out:
        clear_output(wait=True)
        station, view = station_dd.value, view_dd.value
        if view == 'RH time series':
            plot_rh_timeseries(station)
        elif view == 'Valid vs flagged':
            plot_valid_vs_flagged(station)
        elif view == 'Gap timeline':
            plot_gap_timeline(station)
        elif view == 'Seasonal heatmap':
            plot_seasonal_heatmap(station)
        elif view == 'Flag summary':
            show_flag_summary(station)

station_dd.observe(render, names='value')
view_dd.observe(render, names='value')

display(widgets.HBox([station_dd, view_dd]), out)
render()


Output()